## Using Amazon Personalize to recommend related items during hotel search

In [413]:
import pandas as pd
import webbrowser, os
import json
import boto3
import re
import sagemaker
import time
import uuid
from sagemaker import get_execution_role
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

In [314]:
s3 = boto3.client('s3')
personalize = boto3.client('personalize')
personalize_runtime = boto3.client('personalize-runtime')
role = get_execution_role()
print(role)
bucket = 'a2i-experiments'
prefix = 'ent-innov-aws-ai/chapter-03'

arn:aws:iam::841408598787:role/service-role/AmazonSageMaker-ExecutionRole-20220918T193697


In [315]:
raw_df = pd.read_csv('London_hotel_reviews.csv', header=0)

In [316]:
# First we need to rename the reviewer location and property name columns
raw_df.rename(columns = {'Location Of The Reviewer':'reviewer_loc', 'Property Name':'property_name', 'Review Rating':'rating','Date Of Review':'review_date'}, inplace = True)

In [317]:
# remove nulls
raw_df['review_date'].dropna(inplace=True)

In [318]:
raw_df.shape

(27331, 6)

In [319]:
raw_df.head()

,property_name,rating,Review Title,Review Text,reviewer_loc,review_date
0,Apex London Wall Hotel,5.0,Ottima qualità prezzo,Siamo stati a Londra per un week end ed abbiam...,"Casale Monferrato, Italy",10/20/12
1,Corinthia Hotel London,5.0,"By far, my best hotel in the world",I had a pleasure of staying in this hotel for ...,"Savannah, Georgia",3/23/16
2,The Savoy,5.0,First visit to the American Bar at the Savoy,A very lovely first visit to this iconic hotel...,London,7/30/13
3,Rhodes Hotel,4.0,Nice stay,3 of us stayed at the Rhodes Hotel for 4 night...,"Maui, Hawaii",6/2/12
4,The Savoy,5.0,Perfection,Form the moment we arrived until we left we ex...,"London, United Kingdom",11/24/17


In [320]:
import datetime
# Convert date to unix timestamp
for i, r in raw_df.iterrows():
    if str(r['review_date']) == 'nan':
        r['review_date'] = '11/24/16'
    a = datetime.datetime.strptime(str(r['review_date']),"%m/%d/%y")
    b = datetime.datetime.timestamp(a)
    raw_df.at[i,'timestamp'] = b
raw_df = raw_df.drop(['review_date'], axis=1)
raw_df.head()

,property_name,rating,Review Title,Review Text,reviewer_loc,timestamp
0,Apex London Wall Hotel,5.0,Ottima qualità prezzo,Siamo stati a Londra per un week end ed abbiam...,"Casale Monferrato, Italy",1.350691e+09
1,Corinthia Hotel London,5.0,"By far, my best hotel in the world",I had a pleasure of staying in this hotel for ...,"Savannah, Georgia",1.458691e+09
2,The Savoy,5.0,First visit to the American Bar at the Savoy,A very lovely first visit to this iconic hotel...,London,1.375142e+09
3,Rhodes Hotel,4.0,Nice stay,3 of us stayed at the Rhodes Hotel for 4 night...,"Maui, Hawaii",1.338595e+09
4,The Savoy,5.0,Perfection,Form the moment we arrived until we left we ex...,"London, United Kingdom",1.511482e+09


## Build hotel metadata dataset

In [321]:
# first create a new dataframe with only unique property names
hotel_df = pd.read_csv('properties.csv', header=0)
hotel_df.head()

,property_name,price,property_description
0,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...
1,Corinthia Hotel London,1010,Forbes Travel Guide - Perched on Whitehall Pla...
2,The Savoy,700,"Welcome to The Savoy, a place where history an..."
3,Rhodes Hotel,104,"Modern hotel rooms with air conditioning, in t..."
4,Mondrian London at Sea Containers,277,With a design reminiscent of a 1920s transatla...


In [322]:
# get a randomly generated property ID as the Item ID for the hotel metadata dataset
for idx, row in hotel_df.iterrows():
    a = str(uuid.uuid4())
    hotel_df.at[idx, "property_id"] = a
    # clean up the description a bit
    hotel_df.at[idx, "property_description"] = str(row["property_description"]).strip()
hotel_df.head()

,property_name,price,property_description,property_id
0,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...,92f823bb-91ee-483c-8d1b-656ad744f952
1,Corinthia Hotel London,1010,Forbes Travel Guide - Perched on Whitehall Pla...,ebead615-7734-4520-b03b-9007e134829e
2,The Savoy,700,"Welcome to The Savoy, a place where history an...",3b64bb4c-b2b6-49fb-a6a1-c6ed3fb9551a
3,Rhodes Hotel,104,"Modern hotel rooms with air conditioning, in t...",79039534-2f93-435f-8bf0-af5d2faa2d52
4,Mondrian London at Sea Containers,277,With a design reminiscent of a 1920s transatla...,36ac7f94-b20d-4cd5-9dc9-49561dc1cf08


In [421]:
# create a dataset based on the schema
meta_df = hotel_df.rename(columns={'property_id':'ITEM_ID','property_name':'BRAND','property_description':'DESCRIPTION','price':'PRICE'})
meta_df['PRICE'] = meta_df['PRICE'].astype('int')
meta_df['ITEM_ID'] = meta_df['ITEM_ID'].astype('str')
meta_df.head()

,BRAND,PRICE,DESCRIPTION,ITEM_ID
0,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...,92f823bb-91ee-483c-8d1b-656ad744f952
1,Corinthia Hotel London,1010,Forbes Travel Guide - Perched on Whitehall Pla...,ebead615-7734-4520-b03b-9007e134829e
2,The Savoy,700,"Welcome to The Savoy, a place where history an...",3b64bb4c-b2b6-49fb-a6a1-c6ed3fb9551a
3,Rhodes Hotel,104,"Modern hotel rooms with air conditioning, in t...",79039534-2f93-435f-8bf0-af5d2faa2d52
4,Mondrian London at Sea Containers,277,With a design reminiscent of a 1920s transatla...,36ac7f94-b20d-4cd5-9dc9-49561dc1cf08


In [422]:
# Load this to the S3 bucket
s3_meta_loc="s3://"+bucket+"/"+prefix+"/hotel-metadata.csv"
meta_df.to_csv(s3_meta_loc,index=False)

## Build interactions dataset

### Encode property name with its unique identifier

In [325]:
# For the interactions dataset we just need property ID
int_df = hotel_df[['property_name','property_id']]
int_df.head()

,property_name,property_id
0,Apex London Wall Hotel,92f823bb-91ee-483c-8d1b-656ad744f952
1,Corinthia Hotel London,ebead615-7734-4520-b03b-9007e134829e
2,The Savoy,3b64bb4c-b2b6-49fb-a6a1-c6ed3fb9551a
3,Rhodes Hotel,79039534-2f93-435f-8bf0-af5d2faa2d52
4,Mondrian London at Sea Containers,36ac7f94-b20d-4cd5-9dc9-49561dc1cf08


In [326]:
# We will now add the property IDs back to the raw dataframe
new_df = pd.merge(raw_df, int_df, on='property_name')
new_df.head()

,property_name,rating,Review Title,Review Text,reviewer_loc,timestamp,property_id
0,Apex London Wall Hotel,5.0,Ottima qualità prezzo,Siamo stati a Londra per un week end ed abbiam...,"Casale Monferrato, Italy",1.350691e+09,92f823bb-91ee-483c-8d1b-656ad744f952
1,Apex London Wall Hotel,5.0,Great customer service and comfy bed,"The hotel staff were very helpful, I booked th...","Lostwithiel, United Kingdom",1.508630e+09,92f823bb-91ee-483c-8d1b-656ad744f952
2,Apex London Wall Hotel,5.0,Surprisingly quiet in Central London,I booked 1 night DBB choosing this hotel as it...,Lincolnshire,1.415837e+09,92f823bb-91ee-483c-8d1b-656ad744f952
3,Apex London Wall Hotel,4.0,Excellent service,It's a family trip by staying at a business di...,"Hong Kong, China",1.485821e+09,92f823bb-91ee-483c-8d1b-656ad744f952
4,Apex London Wall Hotel,5.0,Apex London Wall repeat visit,This is really just to provide another 5-star ...,"Edinburgh, United Kingdom",1.449533e+09,92f823bb-91ee-483c-8d1b-656ad744f952


### Encode reviewer location as the unique ID for the reviewer

In [327]:
# We will use the rankdata function from SciPy which ranks values based on their frequency of occurence in the dataset
from scipy.stats import rankdata
# first get a list of value counts for each designation type
review_counts = new_df['reviewer_loc'].value_counts()
review_counts

London, United Kingdom                   1858
London                                    470
New York City, New York                   308
Paris, France                             217
Manchester, United Kingdom                193
                                         ... 
Cheseaux-sur-Lausanne, Switzerland          1
Lillestrøm, Norge                           1
Petah Tiqva, Central District, Israel       1
Draguignan, France                          1
Landstuhl, Germany                          1
Name: reviewer_loc, Length: 6624, dtype: int64

In this case we will not use one hot encoding because there are 6624 values under consideration. So we will use frequency encoding instead. 

In [328]:
# Run frequency encoding
# how frequently does each value occur across the entire dataset
review_freq = review_counts/len(new_df)
# Finally let us use a ranking of these frequency encoded values and assign this to our raw dataframe
temp_freq = rankdata(new_df.reviewer_loc.map(review_freq))
new_df['reviewer_id'] = temp_freq
new_df.head()

,property_name,rating,Review Title,Review Text,reviewer_loc,timestamp,property_id,reviewer_id
0,Apex London Wall Hotel,5.0,Ottima qualità prezzo,Siamo stati a Londra per un week end ed abbiam...,"Casale Monferrato, Italy",1.350691e+09,92f823bb-91ee-483c-8d1b-656ad744f952,5293.5
1,Apex London Wall Hotel,5.0,Great customer service and comfy bed,"The hotel staff were very helpful, I booked th...","Lostwithiel, United Kingdom",1.508630e+09,92f823bb-91ee-483c-8d1b-656ad744f952,2200.0
2,Apex London Wall Hotel,5.0,Surprisingly quiet in Central London,I booked 1 night DBB choosing this hotel as it...,Lincolnshire,1.415837e+09,92f823bb-91ee-483c-8d1b-656ad744f952,10090.5
3,Apex London Wall Hotel,4.0,Excellent service,It's a family trip by staying at a business di...,"Hong Kong, China",1.485821e+09,92f823bb-91ee-483c-8d1b-656ad744f952,19143.5
4,Apex London Wall Hotel,5.0,Apex London Wall repeat visit,This is really just to provide another 5-star ...,"Edinburgh, United Kingdom",1.449533e+09,92f823bb-91ee-483c-8d1b-656ad744f952,19344.5


In [329]:
# Make the reviewer ID a string
new_df['reviewer_id'] = 'U-' + new_df['reviewer_id'].astype(str)
new_df.head()

,property_name,rating,Review Title,Review Text,reviewer_loc,timestamp,property_id,reviewer_id
0,Apex London Wall Hotel,5.0,Ottima qualità prezzo,Siamo stati a Londra per un week end ed abbiam...,"Casale Monferrato, Italy",1.350691e+09,92f823bb-91ee-483c-8d1b-656ad744f952,U-5293.5
1,Apex London Wall Hotel,5.0,Great customer service and comfy bed,"The hotel staff were very helpful, I booked th...","Lostwithiel, United Kingdom",1.508630e+09,92f823bb-91ee-483c-8d1b-656ad744f952,U-2200.0
2,Apex London Wall Hotel,5.0,Surprisingly quiet in Central London,I booked 1 night DBB choosing this hotel as it...,Lincolnshire,1.415837e+09,92f823bb-91ee-483c-8d1b-656ad744f952,U-10090.5
3,Apex London Wall Hotel,4.0,Excellent service,It's a family trip by staying at a business di...,"Hong Kong, China",1.485821e+09,92f823bb-91ee-483c-8d1b-656ad744f952,U-19143.5
4,Apex London Wall Hotel,5.0,Apex London Wall repeat visit,This is really just to provide another 5-star ...,"Edinburgh, United Kingdom",1.449533e+09,92f823bb-91ee-483c-8d1b-656ad744f952,U-19344.5


In [414]:
# For Personalize we need only a few columns from the current dataset
# reviewer id, property_id, timestamp, rating
# we will add a calcuated column which indicates past or current reviews
pers_df = new_df[['reviewer_id','property_id','timestamp','rating']]
pers_df['event_type'] = 'past'
pers_df.head()

,reviewer_id,property_id,timestamp,rating,event_type
0,U-5293.5,92f823bb-91ee-483c-8d1b-656ad744f952,1.350691e+09,5.0,past
1,U-2200.0,92f823bb-91ee-483c-8d1b-656ad744f952,1.508630e+09,5.0,past
2,U-10090.5,92f823bb-91ee-483c-8d1b-656ad744f952,1.415837e+09,5.0,past
3,U-19143.5,92f823bb-91ee-483c-8d1b-656ad744f952,1.485821e+09,4.0,past
4,U-19344.5,92f823bb-91ee-483c-8d1b-656ad744f952,1.449533e+09,5.0,past


In [415]:
# we need only positive reviews to recommend related items
pos_interactions_df = pers_df[pers_df['rating'] > 3.0]
# now rename the column headings to match the schema
pos_interactions_df.rename(columns={'reviewer_id':'USER_ID', 'property_id':'ITEM_ID','timestamp':'TIMESTAMP','rating':'EVENT_VALUE','event_type':'EVENT_TYPE'},inplace=True)
pos_interactions_df['USER_ID'] = pos_interactions_df['USER_ID'].astype('str')
pos_interactions_df['ITEM_ID'] = pos_interactions_df['ITEM_ID'].astype('str')
pos_interactions_df['TIMESTAMP'] = pos_interactions_df['TIMESTAMP'].astype('long')
pos_interactions_df['EVENT_VALUE'] = pos_interactions_df['EVENT_VALUE'].astype('float')
pos_interactions_df['EVENT_TYPE'] = pos_interactions_df['EVENT_TYPE'].astype('str')
pos_interactions_df.head()

,USER_ID,ITEM_ID,TIMESTAMP,EVENT_VALUE,EVENT_TYPE
0,U-5293.5,92f823bb-91ee-483c-8d1b-656ad744f952,1350691200,5.0,past
1,U-2200.0,92f823bb-91ee-483c-8d1b-656ad744f952,1508630400,5.0,past
2,U-10090.5,92f823bb-91ee-483c-8d1b-656ad744f952,1415836800,5.0,past
3,U-19143.5,92f823bb-91ee-483c-8d1b-656ad744f952,1485820800,4.0,past
4,U-19344.5,92f823bb-91ee-483c-8d1b-656ad744f952,1449532800,5.0,past


#### Let us check the distribution of number of reviews/interactions for each property type (ITEM_ID) and normalize the dataset

In [409]:
# create a temporary dataframe with a combination of interactions and meta data
temp_df = pd.merge(pos_interactions_df, meta_df, on='ITEM_ID')
temp_df.head()

,USER_ID,ITEM_ID,TIMESTAMP,EVENT_VALUE,EVENT_TYPE,BRAND,PRICE,DESCRIPTION
0,U-5293.5,92f823bb-91ee-483c-8d1b-656ad744f952,1350691200,5.0,past,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...
1,U-2200.0,92f823bb-91ee-483c-8d1b-656ad744f952,1508630400,5.0,past,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...
2,U-10090.5,92f823bb-91ee-483c-8d1b-656ad744f952,1415836800,5.0,past,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...
3,U-19143.5,92f823bb-91ee-483c-8d1b-656ad744f952,1485820800,4.0,past,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...
4,U-19344.5,92f823bb-91ee-483c-8d1b-656ad744f952,1449532800,5.0,past,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...


In [411]:
temp_df['BRAND'].value_counts()

The Savoy                                                         4947
Mondrian London at Sea Containers                                 4014
Corinthia Hotel London                                            2693
The Rembrandt                                                     2564
Apex London Wall Hotel                                            2122
The Dorchester                                                    1563
Hotel Xenia, Autograph Collection                                 1402
Rhodes Hotel                                                      1245
Ridgemount Hotel                                                  1220
Mandarin Oriental Hyde Park, London                                997
Bulgari Hotel, London                                              441
The Wellesley Knightsbridge, a Luxury Collection Hotel, London     319
The Lanesborough                                                   310
London Guest House                                                 229
45 Par

There are a lot of interactions on some of the popular luxury hotels but not a lot of interactions for the budget hotels. Let us normalize the dataset such that all hotels are considered and the hotel metadata also influences the recommendations

In [416]:
pos_interactions_df.shape

(24347, 5)

In [418]:
# now create an updated interactions datasets containing only 200 rows for every hotel or ITEM_ID
pos_int_updated_df = pos_interactions_df.groupby('ITEM_ID').head(200)
pos_int_updated_df.shape

(3081, 5)

In [419]:
# upload this DF as a CSV file into our S3 bucket
s3_loc = "s3://"+bucket+"/"+prefix+"/hotel-review-interactions.csv"
pos_int_updated_df.to_csv(s3_loc, index=False)
print(s3_loc)

s3://a2i-experiments/ent-innov-aws-ai/chapter-03/hotel-review-interactions.csv


#### That completes our dataset manipulation tasks, we are now ready for trying the Personalize recipe

## Amazon Personalize tasks

### Import Personalize Datasets

In [420]:
# we already declared the Personalize boto3 handle in step 1 of this notebook
# we can directly start creating objects to hold our dataset here
pers_ds_group_response = personalize.create_dataset_group(name="similar-hotels-ds-grp-5")
pers_ds_group_arn = pers_ds_group_response['datasetGroupArn']
print(pers_ds_group_arn)

arn:aws:personalize:us-east-1:841408598787:dataset-group/similar-hotels-ds-grp-5


In [423]:
#check the status of the dataset group before proceeding
pers_ds_status = personalize.describe_dataset_group(datasetGroupArn = pers_ds_group_arn)
print(pers_ds_status['datasetGroup']['status'])

ACTIVE


#### Do not proceed to the next step until the status above is **ACTIVE**

#### Define the interactions schema

In [424]:
# Now we will create the schema for our interactions dataset
hotel_int_schema = {
    "type": "record",
    "name": "Interactions",
    "namespace": "com.amazonaws.personalize.schema",
    "fields": [
        {
            "name": "USER_ID",
            "type": "string"
        },
        {
            "name": "ITEM_ID",
            "type": "string"
        },
        {
            "name": "TIMESTAMP",
            "type": "long"
        },
        {
            "name": "EVENT_VALUE",
            "type": "float"
        },
        {
            "name": "EVENT_TYPE",
            "type": "string"
        }
    ],
    "version": "1.0"
}
            
create_schema_response = personalize.create_schema(
    name = "hotel-review-interactions-schema-5",
    schema = json.dumps(hotel_int_schema)
)

interaction_schema_arn = create_schema_response['schemaArn']
print(json.dumps(create_schema_response, indent=2))

{
  "schemaArn": "arn:aws:personalize:us-east-1:841408598787:schema/hotel-review-interactions-schema-5",
  "ResponseMetadata": {
    "RequestId": "d57e192c-1ec5-488a-ac90-10111ba5469e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 31 Jan 2023 19:28:22 GMT",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "100",
      "connection": "keep-alive",
      "x-amzn-requestid": "d57e192c-1ec5-488a-ac90-10111ba5469e"
    },
    "RetryAttempts": 0
  }
}


#### Create the interactions dataset

In [425]:
dataset_type = "INTERACTIONS"
create_dataset_response = personalize.create_dataset(
    name = "hotel-review-interactions-ds-5",
    datasetType = dataset_type,
    datasetGroupArn = pers_ds_group_arn,
    schemaArn = interaction_schema_arn
)

interactions_dataset_arn = create_dataset_response['datasetArn']
print(interactions_dataset_arn)

arn:aws:personalize:us-east-1:841408598787:dataset/similar-hotels-ds-grp-5/INTERACTIONS


In [427]:
#check the status of the dataset group before proceeding
create_ds_status = personalize.describe_dataset(datasetArn = interactions_dataset_arn)
print(create_ds_status['dataset']['status'])

ACTIVE


#### Upload the interactions dataset

In [428]:
create_dataset_import_job_response = personalize.create_dataset_import_job(
    jobName = "hotel-review-interactions-import-5",
    datasetArn = interactions_dataset_arn,
    dataSource = {
        "dataLocation": s3_loc
    },
    roleArn = role
)

dataset_import_job_arn = create_dataset_import_job_response['datasetImportJobArn']
print(dataset_import_job_arn)

arn:aws:personalize:us-east-1:841408598787:dataset-import-job/hotel-review-interactions-import-5


In [430]:
#check the status of the dataset import before proceeding
ds_import_status = personalize.describe_dataset_import_job(datasetImportJobArn = dataset_import_job_arn)
print(ds_import_status['datasetImportJob']['status'])

ACTIVE


#### Define metadata schema

In [432]:
item_schema = {
    "type": "record",
    "name": "Items",
    "namespace": "com.amazonaws.personalize.schema",
    "fields": [
        {
            "name": "ITEM_ID",
            "type": "string"
        },
        {
            "name": "BRAND",
            "type": ["null","string"]
        },
        {
            "name": "PRICE",
            "type": ["null","int"]
        },
        {
            "name": "DESCRIPTION",
            "type": [ "null", "string" ],
            "textual": True
        }
    ],
    "version": "1.0"
}

create_schema_response = personalize.create_schema(
    name = "hotel-metadata-schema-5",
    schema = json.dumps(item_schema)
)

item_schema_arn = create_schema_response['schemaArn']
print(json.dumps(create_schema_response, indent=2))

{
  "schemaArn": "arn:aws:personalize:us-east-1:841408598787:schema/hotel-metadata-schema-5",
  "ResponseMetadata": {
    "RequestId": "18384528-0048-4c34-882f-7e63d6f8acbd",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 31 Jan 2023 19:41:31 GMT",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "89",
      "connection": "keep-alive",
      "x-amzn-requestid": "18384528-0048-4c34-882f-7e63d6f8acbd"
    },
    "RetryAttempts": 0
  }
}


In [433]:
dataset_type = "ITEMS"
create_dataset_response = personalize.create_dataset(
    name = "hotel-metadata-items-5",
    datasetType = dataset_type,
    datasetGroupArn = pers_ds_group_arn,
    schemaArn = item_schema_arn
)

items_dataset_arn = create_dataset_response['datasetArn']
print(items_dataset_arn)

arn:aws:personalize:us-east-1:841408598787:dataset/similar-hotels-ds-grp-5/ITEMS


In [436]:
#check the status of the dataset before proceeding
create_item_ds_status = personalize.describe_dataset(datasetArn = items_dataset_arn)
print(create_item_ds_status['dataset']['status'])

ACTIVE


In [437]:
create_dataset_import_job_response = personalize.create_dataset_import_job(
    jobName = "hotel-metadata-import-job-5",
    datasetArn = items_dataset_arn,
    dataSource = {
        "dataLocation": s3_meta_loc
    },
    roleArn = role
)

dataset_import_job_items_arn = create_dataset_import_job_response['datasetImportJobArn']
print(dataset_import_job_items_arn)

arn:aws:personalize:us-east-1:841408598787:dataset-import-job/hotel-metadata-import-job-5


In [439]:
describe_dataset_import_job_response = personalize.describe_dataset_import_job(datasetImportJobArn = dataset_import_job_items_arn)
print(describe_dataset_import_job_response["datasetImportJob"]['status'])

ACTIVE


### Create Solution

In [477]:
# Get the list of available recipes
for recipe in personalize.list_recipes()['recipes']:
    print(recipe['name'])

aws-ecomm-customers-who-viewed-x-also-viewed
aws-ecomm-frequently-bought-together
aws-ecomm-popular-items-by-purchases
aws-ecomm-popular-items-by-views
aws-ecomm-recommended-for-you
aws-item-affinity
aws-item-attribute-affinity
aws-personalized-ranking
aws-popularity-count
aws-similar-items
aws-sims


In [440]:
similar_recipe_arn = "arn:aws:personalize:::recipe/aws-similar-items"
print(similar_recipe_arn)

arn:aws:personalize:::recipe/aws-similar-items


In [441]:
similar_sol_response = personalize.create_solution(
    name = "hotel-similar-items-recommendation-6",
    datasetGroupArn = pers_ds_group_arn,
    recipeArn = similar_recipe_arn
)

similar_sol_arn = similar_sol_response['solutionArn']

In [442]:
describe_sol_response = personalize.describe_solution(solutionArn = similar_sol_arn)
print(describe_sol_response["solution"]['status'])

ACTIVE


### Create Solution Version

In [443]:
similar_sol_version_response = personalize.create_solution_version(solutionArn=similar_sol_arn)
print(similar_sol_version_response['solutionVersionArn'])

arn:aws:personalize:us-east-1:841408598787:solution/hotel-similar-items-recommendation-6/7e7b620a


In [446]:
describe_sol_ver_response = personalize.describe_solution_version(solutionVersionArn = similar_sol_version_response['solutionVersionArn'])
print(describe_sol_ver_response["solutionVersion"]['status'])

ACTIVE


In [447]:
print("Training time: " + str(describe_sol_ver_response["solutionVersion"]["trainingHours"]))

Training time: 4.473


### Create Campaign for the solution

In [448]:
create_campaign_response = personalize.create_campaign(
        name = "hotel-similar-items-campaign-5",
        solutionVersionArn = similar_sol_version_response['solutionVersionArn'],
        minProvisionedTPS = 1
)

campaign_arn = create_campaign_response['campaignArn']
print(campaign_arn)

arn:aws:personalize:us-east-1:841408598787:campaign/hotel-similar-items-campaign-5


In [451]:
describe_campaign_response = personalize.describe_campaign(
            campaignArn = campaign_arn
)
print(describe_campaign_response["campaign"]["status"])

ACTIVE


In [245]:
# bring up the interactions dataframe
pos_interactions_df.head()

,USER_ID,ITEM_ID,TIMESTAMP,EVENT_VALUE,EVENT_TYPE
0,U-5293.5,23afeb1d-4ac6-4088-aacf-7eb58c53b4d1,1350691200,5.0,past
1,U-2200.0,23afeb1d-4ac6-4088-aacf-7eb58c53b4d1,1508630400,5.0,past
2,U-10090.5,23afeb1d-4ac6-4088-aacf-7eb58c53b4d1,1415836800,5.0,past
3,U-19143.5,23afeb1d-4ac6-4088-aacf-7eb58c53b4d1,1485820800,4.0,past
4,U-19344.5,23afeb1d-4ac6-4088-aacf-7eb58c53b4d1,1449532800,5.0,past


In [412]:
# and the hotel metadata dataframe
meta_df.head()

,BRAND,PRICE,DESCRIPTION,ITEM_ID
0,Apex London Wall Hotel,196,This 4-star boutique hotel adds lots of style ...,92f823bb-91ee-483c-8d1b-656ad744f952
1,Corinthia Hotel London,1010,Forbes Travel Guide - Perched on Whitehall Pla...,ebead615-7734-4520-b03b-9007e134829e
2,The Savoy,700,"Welcome to The Savoy, a place where history an...",3b64bb4c-b2b6-49fb-a6a1-c6ed3fb9551a
3,Rhodes Hotel,104,"Modern hotel rooms with air conditioning, in t...",79039534-2f93-435f-8bf0-af5d2faa2d52
4,Mondrian London at Sea Containers,277,With a design reminiscent of a 1920s transatla...,36ac7f94-b20d-4cd5-9dc9-49561dc1cf08


### Run recommendations

The hotels in our dataset are a combination of luxury, budget, contemporary and boutique hotels. While we attempted to normalize the number of interactions for each hotel to 200, there are still a few hotels for which the interaction count is limited in the underlying dataset. This will influence the recommendations 

In [462]:
# Lets look at the hotel metadata once again and the interaction value counts
temp_df = pd.merge(pos_int_updated_df, meta_df, on='ITEM_ID')
temp_df['BRAND'].value_counts()

Apex London Wall Hotel                                            200
Ridgemount Hotel                                                  200
The Lanesborough                                                  200
Bulgari Hotel, London                                             200
Hotel Xenia, Autograph Collection                                 200
London Guest House                                                200
The Rembrandt                                                     200
Corinthia Hotel London                                            200
The Wellesley Knightsbridge, a Luxury Collection Hotel, London    200
The Dorchester                                                    200
Mandarin Oriental Hyde Park, London                               200
Mondrian London at Sea Containers                                 200
Rhodes Hotel                                                      200
The Savoy                                                         200
45 Park Lane - Dorch

So if we just ask Personalize for the top 5 recommendations it will not mention the hotels with a low interaction count

In [464]:
meta_df[['BRAND','PRICE']]

,BRAND,PRICE
0,Apex London Wall Hotel,196
1,Corinthia Hotel London,1010
2,The Savoy,700
3,Rhodes Hotel,104
4,Mondrian London at Sea Containers,277
5,"Mandarin Oriental Hyde Park, London",772
6,The Dorchester,882
7,A To Z Hotel,69
8,Ridgemount Hotel,207
9,"The Wellesley Knightsbridge, a Luxury Collecti...",531


In [478]:
# get a test property ID or Item ID - we will use the Dorchester hotel as an example
# in real-time when an user is browing this property, Personalize will recommend related properties to this property as part of this run
# get top 5 recommendations
reco = personalize_runtime.get_recommendations(
        campaignArn=campaign_arn,
        itemId=str(meta_df.query('BRAND == "The Dorchester"')['ITEM_ID'].item()),
        numResults=5
)

In [479]:
res_df = pd.DataFrame()
i = -1
for item in reco['itemList']:
    i += 1
    a = item['itemId']
    res_df.at[i, "BRAND"] = meta_df.query('ITEM_ID == @a')['BRAND'].item()
    res_df.at[i, "PRICE"] = meta_df.query('ITEM_ID == @a')['PRICE'].item()
    res_df.at[i, "DESCRIPTION"] = meta_df.query('ITEM_ID == @a')['DESCRIPTION'].item()
res_df

,BRAND,PRICE,DESCRIPTION
0,"The Wellesley Knightsbridge, a Luxury Collecti...",531.0,Overlooking the Royal Hyde Park amidst the gla...
1,The Rembrandt,207.0,A favourite of independent business and leisur...
2,"Mandarin Oriental Hyde Park, London",772.0,"Internationally acclaimed designer, Joyce Wang..."
3,Ridgemount Hotel,207.0,The Ridgemount Hotel is in the Bloomsbury area...
4,"Bulgari Hotel, London",806.0,Located in Knightsbridge on the edge of Hyde P...


## END OF NOTEBOOK - please go back to Chapter 3